In this notebook, we understand how to extract information from:
- `Database`
- matching catalogs
- consolidated catalog
- extract cutouts
for the "Cosmic Horseshoe" (`J1148+1930`) system and some useful numbers related to the database.

In [1]:
import io

import numpy as np
import pandas as pd
from minio import Minio


def get_proper_values_scalar(item):
    """
    Processes a single item from a DataFrame cell.
    If the item contains multiple values separated by " § ", it splits them into a list.
    Otherwise, it returns the item as a string (or NaN if it's NaN).
    This is used to handle fields where multiple references or values
    for a single object are concatenated in the raw CSV.
    """
    if pd.isna(item):
        return item

    value = str(item)

    if " § " in value:
        return value.split(" § ")
    else:
        return value

While it is possible to obtain tabular data by navigating through each folder in a repository and downloading the necessary files, in this notebook you'll obtain all information for this object, including cutouts using our MinIO's service API:

In [2]:
MINIO_ENDPOINT_URL = "1491-152-84-248-250.ngrok-free.app"
ACCESS_KEY = "slcomp"
SECRET_KEY = "slcomp@data"
client = Minio(
    MINIO_ENDPOINT_URL,
    access_key=ACCESS_KEY,
    secret_key=SECRET_KEY,
    secure=True,
)

In [3]:
client.list_buckets()

[Bucket('slcomp')]

# 1. Obtain data from the database

You can obtain the object and load in the memory using `get_object` method:

In [4]:
Database_object = client.get_object("slcomp", "Data/Database.csv").data
Database = pd.read_csv(
    io.StringIO(Database_object.decode("utf-8")), low_memory=False, dtype=object
)
Database.head()

,JNAME,Original_ID,Alternative_Name,RA,DEC,Individual_Coordinates,Grade,z_L,z_LErr,z_LType,...,mag_yErr,mag_F814W,mag_F814WErr,mag_F814WS,System_Type,Lens_Type,Source_Type,Other_Matches,Original_Comments,Reference
0,J000001.1-350334.9,MS2357.4-3520,NaN,0.00471,-35.05971,00:00:01.13-35:03:34.94,NaN,0.508,NaN,NS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Gioia et al. (1990) - doi:10.1086/191426
1,J000001.2+051908.6,NSCS J000001+051909,NaN,0.00504,5.31906,00:00:01.21+05:19:08.62,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Lopes et al. (2004) - doi:10.1086/423038
2,J000002.2+085518.4,NSCS J000002+085519,NaN,0.00925,8.92179,00:00:02.22+08:55:18.44,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Lopes et al. (2004) - doi:10.1086/423038
3,J000006.0+081629.7,NSCS J000006+081630,NaN,0.02512,8.27492,00:00:06.03+08:16:29.71,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Lopes et al. (2004) - doi:10.1086/423038
4,J000006.1+213829.4,DESI-000.0254+21.6415,NaN,0.02542,21.6415,00:00:06.10+21:38:29.40,C,0.502973,NaN,S,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Huang et al. (2021) - arXiv:2005.04730


Or you can just download the data from the client and save it in another place:

In [5]:
# client.fget_object(
#     bucket_name="slcomp", object_name="Data/Database.csv", file_path="Database.csv"
# )

### Finding a Specific Object (e.g., "Cosmic Horseshoe")

We'll demonstrate how to find the "Cosmic Horseshoe" (approx. `J1148+1930`). Its `Original_ID` might vary across different source catalogs. 
We can search for parts of its name in `Original_ID` or `Alternative_Name`. Note: The `JNAME` in our database has higher precision coordinates.

In [6]:
search_pattern = r"J1148.*1930"

horseshoe_candidates = Database[
    Database.Original_ID.astype(str).str.contains(search_pattern, regex=True, na=False)
]  # You also might try Alternative_Name

if not horseshoe_candidates.empty:
    cosmic_horseshoe_data = horseshoe_candidates.iloc[0]
    print(f"Found Cosmic Horseshoe with JNAME: {cosmic_horseshoe_data['JNAME']}")
    display(cosmic_horseshoe_data)
    TARGET_JNAME = cosmic_horseshoe_data["JNAME"]
else:
    print(
        "Cosmic Horseshoe not found with this search pattern. Using a known demo JNAME."
    )
    TARGET_JNAME = "J114833.1+193003.2"
    cosmic_horseshoe_data = Database[Database.JNAME == TARGET_JNAME].iloc[
        0
    ]  # Assumes unique JNAME
    display(cosmic_horseshoe_data)

Found Cosmic Horseshoe with JNAME: J114833.1+193003.2


JNAME                                                    J114833.1+193003.2
Original_ID               SDSS J114833.14+193003.2 § SDSSJ114833.14+1930...
Alternative_Name                          nan § CosmicHorseshoe,CSWA1 § nan
RA                                                                177.13808
DEC                                                                19.50088
Individual_Coordinates    11:48:33.14+19:30:03.20 § 11:48:33.14+19:30:03...
Grade                                                         nan § A § nan
z_L                                                 0.4457 § 0.4457 § 0.444
z_LErr                                                0.0003 § 0.0003 § nan
z_LType                                                           S § S § S
z_S                                                 2.3811 § 2.3811 § 2.379
z_SErr                                                0.0003 § 0.0003 § nan
z_SType                                                           S § S § S
velDisp     

The `Database` contains some columns where multiple pieces of information (e.g., from different original papers) for the same object are stored in a single string, separated by " § ". The `get_proper_values_scalar` function helps parse these into lists for easier handling. We apply this to all columns:

In [7]:
for key in Database.keys():
    Database[key] = Database[key].apply(get_proper_values_scalar)

# 2. Obtain data from matched catalogs (photometric and spectroscopic)

Since we already have the `JNAME`, our query will be easier:

In [8]:
Photometric_Match_object = client.get_object(
    "slcomp", "Data/Photometric_Match.csv"
).data
Photometric_Match = pd.read_csv(
    io.StringIO(Photometric_Match_object.decode("utf-8")),
    low_memory=False,
    dtype=object,
)
Photometric_Match.head()

,JNAME,RA,DEC,Matching_Index,Catalog_Name,Survey_ID,mag_u,mag_g,mag_r,mag_i,...,galdepth_r,galdepth_z,brickname,Spec-z_L,mag_y,mag_yErr,tilename,System_ID,object_id,Photo-z_LRisk
0,J000001.2+051908.6,0.00504,5.31906,4287,HSC Wide DR3,NaN,NaN,24.6800842,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.484305345786723e+16,4.484305345786723e+16,0.229601771
1,J000006.1+213829.4,0.02542,21.6415,0 § 0,Legacy DR9 § DELVE DR2,9906629443846747 § 10390400122247,NaN,20.961098 § 20.720509,19.652534 § 19.648558,NaN,...,509.99063 § nan,82.645744 § nan,0001p217 § nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,J000008.9+211702.8,0.03692,21.2841,1 § 1,Legacy DR9 § DELVE DR2,9906629267817254 § 10390400063395,NaN,21.433722 § 21.26742,19.858047 § 19.825077,NaN,...,374.18472 § nan,72.87822 § nan,0001p212 § nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,J000014.7-055702.8,0.06112,-5.95079,2 § 2,Legacy DR9 § DELVE DR2,9906619155220480 § 10672000231000,NaN,20.73056 § 20.598766,19.552368 § 19.763075,nan § 19.342503,...,1412.5354 § nan,236.78351 § nan,0001m060 § nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,J000018.9+001617.7,0.07883,0.27158,3 § 3 § 0 § 254,Legacy DR9 § DELVE DR2 § DES DR2 § HSC Wide DR3,9906621511633577 § 10595200353252 § 1032701860...,NaN,21.80206 § 21.608852 § 21.885105 § 22.3174744,20.478874 § 20.651608 § 20.7814 § 21.018671,nan § 20.061157 § 20.031702 § 19.8634529,...,1840.6876 § nan § nan § nan,196.92252 § nan § nan § nan,0001p002 § nan § nan § nan,NaN,nan § nan § 19.58907 § 19.2229767,nan § nan § 0.06786754 § 0.00791850127,nan § nan § DES2359+0001 § nan,nan § nan § nan § 4.164565646445266e+16,nan § nan § nan § 4.164565646445266e+16,nan § nan § nan § 0.0161394048


In [9]:
Photometric_Match.query(f'JNAME=="{TARGET_JNAME}"')

,JNAME,RA,DEC,Matching_Index,Catalog_Name,Survey_ID,mag_u,mag_g,mag_r,mag_i,...,galdepth_r,galdepth_z,brickname,Spec-z_L,mag_y,mag_yErr,tilename,System_ID,object_id,Photo-z_LRisk
10055,J114833.1+193003.2,177.13808,19.50088,2711 § 9602 § 9122,SDSS DR17 § Legacy DR9 § DELVE DR2,1237668293376278947 § 9906628690904106 § 10409...,24.813679 § nan § nan,20.749208 § 20.414946 § 20.52028,18.931202 § 18.522522 § 18.751598,18.152424 § nan § 18.081078,...,nan § 320.54712 § nan,nan § 51.832115 § nan,nan § 1772p195 § nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
Spectroscopic_Match_object = client.get_object(
    "slcomp", "Data/Spectroscopic_Match.csv"
).data
Spectroscopic_Match = pd.read_csv(
    io.StringIO(Spectroscopic_Match_object.decode("utf-8")),
    low_memory=False,
    dtype=object,
)
Spectroscopic_Match.head()

,JNAME,RA,DEC,Matching_Index,Catalog_Name,Spec-z_L,velDisp,Spec-z_LErr,velDispErr,Survey_ID,...,Spec-z_LType,mag_r,mag_u,mag_g,mag_z,mag_uErr,mag_gErr,mag_rErr,mag_iErr,mag_zErr
0,J000006.9-633539.7,0.02858,-63.59437,57421,CfA,0.13599741725323858,NaN,NaN,NaN,A2357-6352,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,J000020.8-002051.0,0.08679,-0.3475,0 § 1499,SDSS DR17 § HSC Wide DR3,0.5598464 § 0.559846401,316.18295 § nan,0.00018708536 § 0.000187085403,46.32333 § nan,4746869140807243776 § 40576952637090352,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,J000025.6+283211.9,0.10646,28.53665,1,SDSS DR17,0.35074893,204.7582,5.9427417e-05,19.321684,7330987840606853120,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,J000101.1-003328.3,0.25442,-0.55785,2 § 1500,SDSS DR17 § HSC Wide DR3,0.57276654 § 0.572766483,255.17322 § nan,0.0 § 0.000140991004,38.168335 § nan,1732801237692213248 § 40576810903164582,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,J000106.0+234605.3,0.27508,23.76815,3,SDSS DR17,0.6386577,382.85495,0.00015473602,58.976784,7745239016837240832,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
Spectroscopic_Match.query(f'JNAME=="{TARGET_JNAME}"')

,JNAME,RA,DEC,Matching_Index,Catalog_Name,Spec-z_L,velDisp,Spec-z_LErr,velDispErr,Survey_ID,...,Spec-z_LType,mag_r,mag_u,mag_g,mag_z,mag_uErr,mag_gErr,mag_rErr,mag_iErr,mag_zErr
3158,J114833.1+193003.2,177.13808,19.50088,2669,SDSS DR17,0.44546556,257.88397,0.0,41.802395,2831723547989141504,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 3. Consolidated catalog

In [12]:
Consolidated_Catalog_object = client.get_object(
    "slcomp", "Data/Consolidated_Data.csv"
).data
Consolidated_Catalog = pd.read_csv(
    io.StringIO(Consolidated_Catalog_object.decode("utf-8")),
    low_memory=False,
    dtype=object,
)
Consolidated_Catalog.head()

,JNAME,RA,DEC,System_Type,Lens_Type,Source_Type,theta_E,theta_EErr,theta_EMethod,theta_ERef,...,mag_zS,mag_zSRef,mag_y,mag_yErr,mag_yRef,mag_F814W,mag_F814WErr,mag_F814WRef,mag_F814WS,mag_F814WSRef
0,J000001.1-350334.9,0.00471,-35.05971,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,J000001.2+051908.6,0.00504,5.31906,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,J000002.2+085518.4,0.00925,8.92179,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,J000006.0+081629.7,0.02512,8.27492,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,J000006.1+213829.4,0.02542,21.6415,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
Consolidated_Catalog.query(f'JNAME=="{TARGET_JNAME}"')

,JNAME,RA,DEC,System_Type,Lens_Type,Source_Type,theta_E,theta_EErr,theta_EMethod,theta_ERef,...,mag_zS,mag_zSRef,mag_y,mag_yErr,mag_yRef,mag_F814W,mag_F814WErr,mag_F814WRef,mag_F814WS,mag_F814WSRef
16123,J114833.1+193003.2,177.13808,19.50088,Non Single Lens Galaxy,GROUP,GALAXY,5.1,0.1,SIE model,Moustakas et al. (2012) - adsabs:2012hst..prop...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# 4. Cutouts

For this task we will save all cutouts for the "Cosmic Horseshoe" (including FITS files from all sizes and processed cutouts):

In [14]:
Cutouts_object_content = client.get_object("slcomp", "Cutouts/FITS.parquet").data
Cutouts_Catalog = pd.read_parquet(io.BytesIO(Cutouts_object_content))
Cutouts_Catalog.head()

,JNAME,survey,cutout_size,band,tile,file_name,file_path
0,J020041.8-110240.2,CFHTLenS,20asec,y,W1m4m4,J020041.8-110240.2_W1m4m4_y.V2.2A.swarp.cut.19...,FITS/20asec/CFHTLenS/J020041.8-110240.2_W1m4m4...
1,J020041.8-110240.2,CFHTLenS,20asec,z,W1m4m4,J020041.8-110240.2_W1m4m4_z.V2.2A.swarp.cut.19...,FITS/20asec/CFHTLenS/J020041.8-110240.2_W1m4m4...
2,J020041.8-110240.2,CFHTLenS,20asec,g,W1m4m4,J020041.8-110240.2_W1m4m4_g.V2.2A.swarp.cut.19...,FITS/20asec/CFHTLenS/J020041.8-110240.2_W1m4m4...
3,J020041.8-110240.2,CFHTLenS,20asec,r,W1m4m4,J020041.8-110240.2_W1m4m4_r.V2.2A.swarp.cut.19...,FITS/20asec/CFHTLenS/J020041.8-110240.2_W1m4m4...
4,J020041.8-110240.2,CFHTLenS,20asec,u,W1m4m4,J020041.8-110240.2_W1m4m4_u.V2.2A.swarp.cut.19...,FITS/20asec/CFHTLenS/J020041.8-110240.2_W1m4m4...


In [15]:
Cutouts_Catalog.query(f'JNAME=="{TARGET_JNAME}"')

,JNAME,survey,cutout_size,band,tile,file_name,file_path
140938,J114833.1+193003.2,Legacy,20asec,g,1772p195,J114833.1+193003.2_legacysurvey-1772p195-image...,FITS/20asec/Legacy/J114833.1+193003.2_legacysu...
140939,J114833.1+193003.2,Legacy,20asec,z,1772p195,J114833.1+193003.2_legacysurvey-1772p195-image...,FITS/20asec/Legacy/J114833.1+193003.2_legacysu...
140940,J114833.1+193003.2,Legacy,20asec,r,1772p195,J114833.1+193003.2_legacysurvey-1772p195-image...,FITS/20asec/Legacy/J114833.1+193003.2_legacysu...
323857,J114833.1+193003.2,Legacy,4amin,z,1772p195,J114833.1+193003.2_legacysurvey-1772p195-image...,FITS/4amin/Legacy/J114833.1+193003.2_legacysur...
323858,J114833.1+193003.2,Legacy,4amin,r,1772p195,J114833.1+193003.2_legacysurvey-1772p195-image...,FITS/4amin/Legacy/J114833.1+193003.2_legacysur...
323859,J114833.1+193003.2,Legacy,4amin,g,1772p195,J114833.1+193003.2_legacysurvey-1772p195-image...,FITS/4amin/Legacy/J114833.1+193003.2_legacysur...


In [16]:
Cutouts_Catalog.query('JNAME=="J114833.1+193003.2"').file_path

140938    FITS/20asec/Legacy/J114833.1+193003.2_legacysu...
140939    FITS/20asec/Legacy/J114833.1+193003.2_legacysu...
140940    FITS/20asec/Legacy/J114833.1+193003.2_legacysu...
323857    FITS/4amin/Legacy/J114833.1+193003.2_legacysur...
323858    FITS/4amin/Legacy/J114833.1+193003.2_legacysur...
323859    FITS/4amin/Legacy/J114833.1+193003.2_legacysur...
Name: file_path, dtype: object

In [17]:
[
    client.fget_object("slcomp", "Cutouts/" + file_path, file_path)
    for file_path in Cutouts_Catalog.query('JNAME=="J114833.1+193003.2"').file_path
]

In [18]:
Processed_Cutouts_object_content = client.get_object(
    "slcomp", "Cutouts/Processed_Cutouts.parquet"
).data
Processed_Cutouts_Catalog = pd.read_parquet(
    io.BytesIO(Processed_Cutouts_object_content)
)
Processed_Cutouts_Catalog.head()

,JNAME,survey,cutout_size,processing,is_rgb,file_name,file_path
0,J000001.1-350334.9,DES,20asec,lsb,True,J000001.1-350334.9.jpeg,Processed_Cutouts_Legacy/20asec/DES/J000001.1-...
1,J000001.1-350334.9,Legacy,20asec,trilogy,False,J000001.1-350334.9_legacysurvey-0001m350-image...,Processed_Cutouts/20asec/Legacy/J000001.1-3503...
2,J000001.1-350334.9,Legacy,20asec,trilogy,False,J000001.1-350334.9_legacysurvey-0001m350-image...,Processed_Cutouts/20asec/Legacy/J000001.1-3503...
3,J000001.1-350334.9,Legacy,20asec,lsb,True,J000001.1-350334.9.jpeg,Processed_Cutouts_Legacy/20asec/Legacy/J000001...
4,J000001.1-350334.9,DES,4amin,lsb,True,J000001.1-350334.9.jpeg,Processed_Cutouts_Legacy/4amin/DES/J000001.1-3...


In [19]:
Processed_Cutouts_Catalog.query('JNAME=="J114833.1+193003.2"')

,JNAME,survey,cutout_size,processing,is_rgb,file_name,file_path
340283,J114833.1+193003.2,Legacy,20asec,trilogy,False,J114833.1+193003.2_legacysurvey-1772p195-image...,Processed_Cutouts/20asec/Legacy/J114833.1+1930...
340284,J114833.1+193003.2,Legacy,20asec,trilogy,True,J114833.1+193003.2_Legacy_1772p195_20asec.png,Processed_Cutouts/20asec/Legacy/J114833.1+1930...
340285,J114833.1+193003.2,Legacy,20asec,trilogy,False,J114833.1+193003.2_legacysurvey-1772p195-image...,Processed_Cutouts/20asec/Legacy/J114833.1+1930...
340286,J114833.1+193003.2,Legacy,20asec,trilogy,False,J114833.1+193003.2_legacysurvey-1772p195-image...,Processed_Cutouts/20asec/Legacy/J114833.1+1930...
340287,J114833.1+193003.2,Legacy,20asec,lsb,True,J114833.1+193003.2.jpeg,Processed_Cutouts_Legacy/20asec/Legacy/J114833...
340288,J114833.1+193003.2,SDSS,20asec,lsb,True,J114833.1+193003.2.jpeg,Processed_Cutouts_Legacy/20asec/SDSS/J114833.1...
340289,J114833.1+193003.2,Legacy,4amin,trilogy,True,J114833.1+193003.2_Legacy_1772p195_4amin.png,Processed_Cutouts/4amin/Legacy/J114833.1+19300...
340290,J114833.1+193003.2,Legacy,4amin,trilogy,False,J114833.1+193003.2_legacysurvey-1772p195-image...,Processed_Cutouts/4amin/Legacy/J114833.1+19300...
340291,J114833.1+193003.2,Legacy,4amin,trilogy,False,J114833.1+193003.2_legacysurvey-1772p195-image...,Processed_Cutouts/4amin/Legacy/J114833.1+19300...
340292,J114833.1+193003.2,Legacy,4amin,trilogy,False,J114833.1+193003.2_legacysurvey-1772p195-image...,Processed_Cutouts/4amin/Legacy/J114833.1+19300...


In [20]:
[
    client.fget_object("slcomp", "Cutouts/" + file_path, file_path)
    for file_path in Processed_Cutouts_Catalog.query(
        'JNAME=="J114833.1+193003.2"'
    ).file_path
]

**To query for a different object, replace TARGET_JNAME with its identifier.**

---

# 5. Summary Statistics of the Catalogs

The following cells provide some summary numbers and distributions for the main database and the cross-matched catalogs to give an overview of their contents.

In [21]:
for key in Photometric_Match.keys():
    Photometric_Match[key] = Photometric_Match[key].apply(get_proper_values_scalar)

for key in Spectroscopic_Match.keys():
    Spectroscopic_Match[key] = Spectroscopic_Match[key].apply(get_proper_values_scalar)

## Unique systems

In [22]:
Database.JNAME.unique().shape[0]

31569

## Systems types

In [23]:
Database.System_Type.value_counts()

System_Type
Single Lens Galaxy               4167
Non Single Lens Galaxy            389
Non Single Lens Galaxy Merged     269
Name: count, dtype: int64

## Lens types

In [24]:
pd.DataFrame(np.hstack(Database.Lens_Type), columns=["Lens_Type"]).value_counts()

Lens_Type    
nan              30203
GALAXY            6660
CLUSTER           4480
GROUP              109
X-RAY CLUSTER       23
QUASAR               3
Name: count, dtype: int64

## Source types

In [25]:
pd.DataFrame(np.hstack(Database.Source_Type), columns=["Source_Type"]).value_counts()

Source_Type
nan            30203
GALAXY         10421
QUASAR           831
CLUSTER           23
Name: count, dtype: int64

## Numbers for Photometric Matches

### Matches per catalog

In [26]:
pd.DataFrame(
    np.hstack(Photometric_Match.Catalog_Name), columns=["Catalog_Name"]
).value_counts()

Catalog_Name
Legacy DR9      15806
DELVE DR2       14837
DES DR2          6750
SDSS DR17        5652
HSC Wide DR3     4640
KiDS DR3         2393
CFHTLS Wide       615
CS82              406
HSC Deep DR3      321
CFHTLS Deep       134
Name: count, dtype: int64

## Numbers for Spectroscopic Matches

In [27]:
pd.DataFrame(
    np.hstack(Spectroscopic_Match.Catalog_Name), columns=["Catalog_Name"]
).value_counts()

Catalog_Name
SDSS DR17       5568
HSC Wide DR3    2528
GAMA DR3         430
CfA              236
HSC D/UD DR3     189
2MASS            157
2dF              130
LAMOST DR7        75
6dF DR3           67
zCOSMOS           41
VIPERS            36
OzDES DR2         21
MGC               20
DEEP2 DR4          7
WiggleZ DR1        6
SSRS               4
AGES               2
2SLAQ              1
LCRS               1
PSCz               1
Name: count, dtype: int64

## Numbers for Consolidated Catalog

### Systems with $\theta_\mathrm{E}$

In [28]:
Consolidated_Catalog.theta_E.notnull().sum()

np.int64(881)

### Systems with $z_L$

In [29]:
Consolidated_Catalog.z_L.notnull().sum()

np.int64(20999)

Redshift types

In [30]:
Consolidated_Catalog.z_LType.value_counts()

z_LType
P    11567
S     6690
Name: count, dtype: int64

### Systems with $z_S$

In [31]:
Consolidated_Catalog.z_S.notnull().sum()

np.int64(3780)

Redshift types

In [32]:
Consolidated_Catalog.z_SType.value_counts()

z_SType
S    2915
P     524
Name: count, dtype: int64

### Systems with $\sigma_v$

In [33]:
Consolidated_Catalog.velDisp.notnull().sum()

np.int64(5663)

### Magnitudes

In [34]:
for mag in [
    "mag_u",
    "mag_uS",
    "mag_g",
    "mag_gS",
    "mag_r",
    "mag_rS",
    "mag_i",
    "mag_iS",
    "mag_z",
    "mag_zS",
    "mag_y",
    "mag_F814W",
    "mag_F814WS",
]:
    print(f"{mag}: {Consolidated_Catalog[mag].notnull().sum()}")

mag_u: 7529
mag_uS: 33
mag_g: 17409
mag_gS: 48
mag_r: 17345
mag_rS: 43
mag_i: 16098
mag_iS: 1061
mag_z: 17125
mag_zS: 112
mag_y: 564
mag_F814W: 364
mag_F814WS: 66


## Cutouts

In [35]:
len(Cutouts_Catalog)

366905

In [36]:
Cutouts_Catalog.value_counts(subset=["survey", "band", "cutout_size"])

survey    band  cutout_size
Legacy    r     20asec         27448
                4amin          27442
          g     20asec         26868
                4amin          26862
          z     20asec         26718
                4amin          26713
DES       r     20asec          8889
                4amin           8889
          i     4amin           8862
                20asec          8862
          z     4amin           8806
                20asec          8806
          g     4amin           8782
                20asec          8782
          y     20asec          8349
                4amin           8349
HSC       g     20asec          7083
                4amin           7074
          r     20asec          6952
                4amin           6937
          z     20asec          6819
                4amin           6789
          i     20asec          6703
                4amin           6697
          y     20asec          6418
                4amin           6401
KiDS      

### Processed Cutouts

In [37]:
Processed_Cutouts_Catalog.value_counts(
    subset=["survey", "processing", "is_rgb", "cutout_size"]
)

survey    processing  is_rgb  cutout_size
Legacy    trilogy     False   20asec         81034
                              4amin          81017
DES       trilogy     False   4amin          43712
                              20asec         43688
HSC       trilogy     False   20asec         33975
                              4amin          33898
Legacy    lsb         True    20asec         29045
                              4amin          25962
          trilogy     True    20asec         24918
                              4amin          24904
SDSS      lsb         True    20asec         21503
                              4amin          18590
KiDS      trilogy     False   20asec         14946
                              4amin          13977
DES       lsb         True    20asec          9117
          trilogy     True    20asec          8764
                              4amin           8764
          lsb         True    4amin           8572
HSC       trilogy     True    20asec    

In [38]:
len(Processed_Cutouts_Catalog.query('is_rgb=="True"'))

217135

In [39]:
len(Processed_Cutouts_Catalog.query('processing=="lsb"'))

125215

In [40]:
len(Processed_Cutouts_Catalog.query('processing=="trilogy"'))

458849